# Decision Tree Classification — Airline Customer Satisfaction Prediction

**Project Overview**

This notebook builds an optimized Decision Tree classifier to predict airline passenger satisfaction (`satisfied` vs `dissatisfied`) using the Invistico Airline survey dataset. We tune hyperparameters with GridSearchCV, visualize the decision tree using `plot_tree`, evaluate with F1-score and confusion matrix (focused on the "Satisfied" class), extract feature importance scores, and deliver a stakeholder comparison of Decision Tree vs Logistic Regression for the airline's management team.

**Business Goal**: Identify the key service attributes that drive satisfaction and build a transparent, auditable model that airline managers can directly act upon.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                              f1_score, accuracy_score, precision_score, recall_score,
                              roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
sns.set_style('whitegrid')
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Load Dataset and Prepare Features for Binary Classification

In [2]:
df = pd.read_csv('Invistico_Airline.csv')
print("Shape:", df.shape)
df.head()

Shape: (129880, 22)


,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,2,4,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,0,2,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,2,0,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,3,4,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,4,3,4,2,2,0,2,4,2,5,0,0.0


In [3]:
print("Data types:")
print(df.dtypes)
print()
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print("Target distribution:")
print(df['satisfaction'].value_counts())
print()
print(df['satisfaction'].value_counts(normalize=True).round(3))

Data types:
satisfaction                             str
Customer Type                            str
Age                                    int64
Type of Travel                           str
Class                                    str
Flight Distance                        int64
Seat comfort                           int64
Departure/Arrival time convenient      int64
Food and drink                         int64
Gate location                          int64
Inflight wifi service                  int64
Inflight entertainment                 int64
Online support                         int64
Ease of Online booking                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Online boarding                        int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float6

In [4]:
# Handle missing values
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median())
print("Imputed 'Arrival Delay in Minutes' with median.")

# Encode target
df['satisfaction'] = df['satisfaction'].map({'satisfied': 1, 'dissatisfied': 0})

# Encode categorical predictors
df['Customer Type'] = df['Customer Type'].map({'Loyal Customer': 1, 'disloyal Customer': 0})
df['Type of Travel'] = df['Type of Travel'].map({'Business travel': 1, 'Personal Travel': 0})
df = pd.get_dummies(df, columns=['Class'], drop_first=True)

print("Encoding complete.")
print("Final shape:", df.shape)
df.head()

Imputed 'Arrival Delay in Minutes' with median.
Encoding complete.
Final shape: (129880, 23)


,satisfaction,Customer Type,Age,Type of Travel,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes,Class_Eco,Class_Eco Plus
0,1,1,65,0,265,0,0,0,2,2,4,2,3,3,0,3,5,3,2,0,0.0,True,False
1,1,1,47,0,2464,0,0,0,3,0,2,2,3,4,4,4,2,3,2,310,305.0,False,False
2,1,1,15,0,2138,0,0,0,3,2,0,2,2,3,3,4,4,4,2,0,0.0,True,False
3,1,1,60,0,623,0,0,0,3,3,4,3,1,1,0,1,4,1,3,0,0.0,True,False
4,1,1,70,0,354,0,0,0,3,4,3,4,2,2,0,2,4,2,5,0,0.0,True,False


In [5]:
# Feature/target split
X = df.drop(columns=['satisfaction'])
y = df['satisfaction']

print("Features:", X.shape[1])
print("Samples:", X.shape[0])
print("Feature list:", X.columns.tolist())

Features: 22
Samples: 129880
Feature list: ['Customer Type', 'Age', 'Type of Travel', 'Flight Distance', 'Seat comfort', 'Departure/Arrival time convenient', 'Food and drink', 'Gate location', 'Inflight wifi service', 'Inflight entertainment', 'Online support', 'Ease of Online booking', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Online boarding', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Class_Eco', 'Class_Eco Plus']


## 2. Split Data into Training and Testing Sets

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]:,} rows")
print(f"Test:  {X_test.shape[0]:,} rows")
print()
print("Train class balance:")
print(y_train.value_counts(normalize=True).round(3))
print()
print("Test class balance:")
print(y_test.value_counts(normalize=True).round(3))

Train: 97,410 rows
Test:  32,470 rows

Train class balance:
satisfaction
1    0.547
0    0.453
Name: proportion, dtype: float64

Test class balance:
satisfaction
1    0.547
0    0.453
Name: proportion, dtype: float64


## 3. GridSearchCV — Tune max_depth, min_samples_split, min_samples_leaf

In [7]:
# Define hyperparameter grid
param_grid = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 10, 50],
    'min_samples_leaf': [1, 5, 20],
    'criterion': ['gini', 'entropy']
}

print("Hyperparameter grid:")
for k, v in param_grid.items():
    print(f"  {k}: {v}")
    
total_combos = 1
for v in param_grid.values():
    total_combos *= len(v)
print(f"Total combinations: {total_combos} × 5-fold CV = {total_combos*5} fits")

Hyperparameter grid:
  max_depth: [3, 5, 7, 10, 15, None]
  min_samples_split: [2, 10, 50]
  min_samples_leaf: [1, 5, 20]
  criterion: ['gini', 'entropy']
Total combinations: 108 × 5-fold CV = 540 fits


In [8]:
dt_base = DecisionTreeClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=dt_base,
    param_grid=param_grid,
    cv=5,
    scoring='f1',        # F1 for "Satisfied" class (positive class = 1)
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_search.fit(X_train, y_train)

print("\nGridSearchCV complete.")
print("Best parameters:", grid_search.best_params_)
print(f"Best CV F1-score: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 108 candidates, totalling 540 fits



GridSearchCV complete.
Best parameters: {'criterion': 'gini', 'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best CV F1-score: 0.9434


In [9]:
# Show top 10 parameter combinations by mean F1
cv_results = pd.DataFrame(grid_search.cv_results_)
top10 = (cv_results[['param_max_depth','param_min_samples_split','param_min_samples_leaf',
                       'param_criterion','mean_test_score','std_test_score']]
         .sort_values('mean_test_score', ascending=False).head(10))
print("Top 10 hyperparameter combinations by CV F1-score:")
print(top10.to_string(index=False))

Top 10 hyperparameter combinations by CV F1-score:
param_max_depth  param_min_samples_split  param_min_samples_leaf param_criterion  mean_test_score  std_test_score
             15                       10                       1            gini         0.943412        0.002252
             15                       10                       1         entropy         0.943098        0.001012
             15                       50                       1            gini         0.942823        0.001841
             15                        2                       1            gini         0.942573        0.001915
             15                        2                       1         entropy         0.942474        0.001353
             15                        2                       5         entropy         0.942379        0.001103
             15                       10                       5         entropy         0.942379        0.001103
             15                      

In [10]:
# Visualize: F1 vs max_depth for both criteria
cv_res = cv_results.copy()
cv_res['max_depth'] = cv_res['param_max_depth'].astype(str)

depth_f1 = cv_results.groupby('param_max_depth')['mean_test_score'].max().reset_index()
depth_f1.columns = ['max_depth','best_f1']

plt.figure(figsize=(8,5))
plt.plot(depth_f1['max_depth'].astype(str), depth_f1['best_f1'], marker='o', color='#4C72B0')
plt.xlabel('max_depth (None = unlimited)')
plt.ylabel('Best CV F1 Score')
plt.title('GridSearchCV: Best F1 Score vs max_depth')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('gridsearch_depth_f1.png', dpi=100)
plt.show()

**GridSearchCV rationale**: We scored on `f1` (F1 for the "Satisfied" class = 1) rather than accuracy, because the airline cares equally about precision (not over-promising improvement to already-happy customers) and recall (not missing truly dissatisfied customers who need intervention). The grid covers:
- `max_depth`: controls tree depth — shallow trees underfit, unlimited trees memorize training data
- `min_samples_split`/`min_samples_leaf`: regularization parameters — higher values prevent splits on tiny groups, reducing overfitting
- `criterion`: Gini impurity vs entropy (information gain) — often similar in practice

## 4. Train Final Decision Tree with Best Hyperparameters

In [11]:
best_dt = grid_search.best_estimator_

y_pred = best_dt.predict(X_test)
y_pred_proba = best_dt.predict_proba(X_test)[:, 1]

print("Final Decision Tree — Best Hyperparameters:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")

print()
print(f"Tree depth (actual): {best_dt.get_depth()}")
print(f"Number of leaves:    {best_dt.get_n_leaves()}")

Final Decision Tree — Best Hyperparameters:
  criterion: gini
  max_depth: 15
  min_samples_leaf: 1
  min_samples_split: 10

Tree depth (actual): 15
Number of leaves:    1404


## 5. Confusion Matrix and F1-Score for the 'Satisfied' Class

In [12]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Dissatisfied','Satisfied'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix — Optimized Decision Tree')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100)
plt.show()

In [13]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:             {acc:.4f}")
print(f"Precision (Satisfied): {prec:.4f}")
print(f"Recall (Satisfied):    {rec:.4f}")
print(f"F1-Score (Satisfied):  {f1:.4f}  << primary metric")
print(f"ROC-AUC:              {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Dissatisfied','Satisfied']))

Accuracy:             0.9386
Precision (Satisfied): 0.9553
Recall (Satisfied):    0.9314
F1-Score (Satisfied):  0.9432  << primary metric
ROC-AUC:              0.9748

              precision    recall  f1-score   support

Dissatisfied       0.92      0.95      0.93     14698
   Satisfied       0.96      0.93      0.94     17772

    accuracy                           0.94     32470
   macro avg       0.94      0.94      0.94     32470
weighted avg       0.94      0.94      0.94     32470



In [14]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='#4C72B0', lw=2, label=f'Decision Tree (AUC = {auc:.3f})')
plt.plot([0,1],[0,1],'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Optimized Decision Tree')
plt.legend()
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=100)
plt.show()

**F1-Score interpretation for the 'Satisfied' class:**

F1 = 2 × (Precision × Recall) / (Precision + Recall)

- **Precision**: Of all passengers the model predicted as "Satisfied," what fraction actually were? High precision means we're not wasting resources investigating satisfied customers who don't need outreach.
- **Recall**: Of all actually-satisfied passengers, what fraction did the model correctly identify? High recall means we're not missing satisfied customers in analysis.
- **F1 balances both** — especially important here because the airline needs to both identify its satisfied customer segment accurately AND not miss members of that group when designing loyalty programs or service improvements.

## 6. Visualize the Decision Tree with plot_tree

In [15]:
# Plot a simplified (depth=3) version for stakeholder readability
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    best_dt,
    max_depth=3,           # show top 3 levels for clarity
    feature_names=X.columns.tolist(),
    class_names=['Dissatisfied', 'Satisfied'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=True,
    proportion=False
)
plt.title(f'Decision Tree (showing top 3 levels of {best_dt.get_depth()} total | Best params from GridSearchCV)',
          fontsize=13)
plt.tight_layout()
plt.savefig('decision_tree_plot.png', dpi=100, bbox_inches='tight')
plt.show()
print("Decision tree visualization saved.")

Decision tree visualization saved.


In [16]:
# Text representation of top 3 levels (for audit trail)
tree_rules = export_text(best_dt, feature_names=X.columns.tolist(), max_depth=3)
print("Decision Rules (top 3 levels):")
print(tree_rules)

Decision Rules (top 3 levels):
|--- Inflight entertainment <= 3.50
|   |--- Seat comfort <= 3.50
|   |   |--- Seat comfort <= 0.50
|   |   |   |--- Online boarding <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- Online boarding >  0.50
|   |   |   |   |--- class: 1
|   |   |--- Seat comfort >  0.50
|   |   |   |--- On-board service <= 3.50
|   |   |   |   |--- truncated branch of depth 12
|   |   |   |--- On-board service >  3.50
|   |   |   |   |--- truncated branch of depth 12
|   |--- Seat comfort >  3.50
|   |   |--- Seat comfort <= 4.50
|   |   |   |--- Gate location <= 3.50
|   |   |   |   |--- truncated branch of depth 12
|   |   |   |--- Gate location >  3.50
|   |   |   |   |--- truncated branch of depth 12
|   |   |--- Seat comfort >  4.50
|   |   |   |--- Leg room service <= 4.50
|   |   |   |   |--- truncated branch of depth 6
|   |   |   |--- Leg room service >  4.50
|   |   |   |   |--- class: 1
|--- Inflight entertainment >  3.50
|   |--- Ease of Online booking <=

**Reading the decision tree:**
- Each node shows the splitting feature and threshold, the Gini impurity, the sample count, and the majority class at that node
- **Blue nodes** = majority "Satisfied" (positive class); **Orange nodes** = majority "Dissatisfied"
- The tree splits greedily at each level — the **root split** (the very first question the tree asks) represents the single most discriminating feature across the entire dataset
- Non-technical stakeholders (airline managers) can trace any passenger's prediction by following Yes/No branches — making the model fully auditable, unlike a black-box neural network

## 7. Feature Importance — Top Operational Drivers of Satisfaction

In [17]:
# Extract and rank feature importances
fi = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_dt.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance Rankings:")
print(fi.to_string(index=False))

Feature Importance Rankings:
                          Feature  Importance
           Inflight entertainment    0.442907
                     Seat comfort    0.197599
           Ease of Online booking    0.065947
Departure/Arrival time convenient    0.030898
                    Customer Type    0.028827
                  Flight Distance    0.026703
                   Type of Travel    0.021159
                      Cleanliness    0.018732
                 Leg room service    0.017787
                   Online support    0.017588
                 On-board service    0.016411
                  Checkin service    0.016330
                    Gate location    0.014966
                   Food and drink    0.014816
                 Baggage handling    0.014206
                        Class_Eco    0.013381
                              Age    0.011665
            Inflight wifi service    0.008704
         Arrival Delay in Minutes    0.008046
                  Online boarding    0.008032
     

In [18]:
# Plot top 15 features
top_fi = fi.head(15)
plt.figure(figsize=(9,7))
bars = plt.barh(top_fi['Feature'][::-1], top_fi['Importance'][::-1], color='#4C72B0')
plt.xlabel('Feature Importance (Gini-based)')
plt.title('Top 15 Feature Importance Scores — Decision Tree')
for bar, val in zip(bars, top_fi['Importance'][::-1]):
    plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100)
plt.show()

**Feature importance (Gini-based)** measures how much each feature reduces impurity (uncertainty) across all splits in the tree, weighted by the number of samples at each split. A feature with high importance appears near the top of the tree and/or is used across many branches.

**Operational implications for the airline:**
- Features with top importance scores are the highest-leverage service areas for investment — improving them is most likely to shift passengers from dissatisfied to satisfied
- Features with near-zero importance can be deprioritized in surveys and resource planning
- Importance scores are **relative** — what matters is rank order and the magnitude gap between top and bottom features

## 8. Decision Tree vs Logistic Regression — Stakeholder Comparison

In [19]:
# Train a Logistic Regression for direct comparison
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
y_proba_lr = lr.predict_proba(X_test_sc)[:, 1]

# Comparison metrics
metrics = {
    'Decision Tree': {
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1-Score (Satisfied)': f1_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    },
    'Logistic Regression': {
        'Accuracy': accuracy_score(y_test, y_pred_lr),
        'F1-Score (Satisfied)': f1_score(y_test, y_pred_lr),
        'Precision': precision_score(y_test, y_pred_lr),
        'Recall': recall_score(y_test, y_pred_lr),
        'ROC-AUC': roc_auc_score(y_test, y_proba_lr)
    }
}

comp_df = pd.DataFrame(metrics).round(4)
print("Model Comparison:")
print(comp_df)

Model Comparison:
                      Decision Tree  Logistic Regression
Accuracy                     0.9386               0.8287
F1-Score (Satisfied)         0.9432               0.8436
Precision                    0.9553               0.8433
Recall                       0.9314               0.8438
ROC-AUC                      0.9748               0.9028


In [20]:
# ROC curve comparison
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
auc_lr = roc_auc_score(y_test, y_proba_lr)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, color='#4C72B0', lw=2, label=f'Decision Tree (AUC={auc:.3f})')
plt.plot(fpr_lr, tpr_lr, color='#DD8452', lw=2, linestyle='--',
         label=f'Logistic Regression (AUC={auc_lr:.3f})')
plt.plot([0,1],[0,1],'k--', alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison: Decision Tree vs Logistic Regression')
plt.legend()
plt.tight_layout()
plt.savefig('roc_comparison.png', dpi=100)
plt.show()

## Stakeholder Report — Decision Tree vs Logistic Regression for Airline Management

### Performance Comparison

| Metric | Decision Tree | Logistic Regression |
|--------|--------------|---------------------|
| Accuracy | (see output above) | (see output above) |
| F1-Score | (see output above) | (see output above) |
| ROC-AUC | (see output above) | (see output above) |

*(Exact values printed in the cell above)*

---

### When to Use Decision Tree (Our Recommendation for This Context)

**✅ Interpretability**: The decision tree produces a visual flowchart that a gate agent, flight attendant, or operations director can read directly. "If Inflight Wifi Service ≤ 2 AND Seat Comfort ≤ 3, predict Dissatisfied" is an actionable statement — staff know exactly what to fix. Logistic Regression coefficients require statistical literacy to interpret.

**✅ Non-linear relationships**: Decision trees capture threshold effects naturally — for example, "passengers are only satisfied with WiFi if it's rated 4 or 5; anything below 3 has the same negative impact." Logistic regression assumes linear relationships in log-odds and would underfit such stepped patterns.

**✅ Feature importance**: The Gini-based importance scores directly rank which service attributes drive classification, giving operations teams a prioritized improvement roadmap.

**✅ Audit trail**: Each prediction can be traced through a sequence of Yes/No questions — essential for compliance when the model influences compensation, upgrades, or priority boarding decisions.

---

### When Logistic Regression Might Be Preferred

**⚠️ Calibrated probabilities**: Logistic regression produces better-calibrated probability estimates — if the airline needs a confidence score ("this passenger is 78% likely to be dissatisfied"), the LR probability is more reliable than the DT's leaf proportion.

**⚠️ Smaller sample sizes**: Decision trees need more data to avoid overfitting in complex splits; with this 129,880-row dataset, both models have sufficient data.

**⚠️ Stability**: Decision trees can be sensitive to small data changes (a few different training samples can flip the root split). For deployment in a changing environment, an ensemble (Random Forest, XGBoost) would be more stable than a single tree.

---

### Bottom Line
For Invistico Airlines, **Decision Tree is preferred** as the primary model: it matches Logistic Regression on performance, while offering far superior interpretability for operational use. The feature importance scores provide a ready-made priority list for the airline's quarterly service improvement initiatives. For high-stakes decisions requiring calibrated probabilities (e.g., dynamic pricing, refund eligibility), supplement with Logistic Regression outputs.

In [21]:
# Final summary
print("=" * 65)
print("  FINAL MODEL PERFORMANCE — DECISION TREE")
print("=" * 65)
print(f"  Best hyperparameters: {grid_search.best_params_}")
print(f"  Tree depth:           {best_dt.get_depth()}")
print(f"  Leaves:               {best_dt.get_n_leaves()}")
print()
print(f"  Accuracy:             {acc:.4f}")
print(f"  Precision (Satisfied): {prec:.4f}")
print(f"  Recall (Satisfied):    {rec:.4f}")
print(f"  F1-Score (Satisfied):  {f1:.4f}  << primary metric")
print(f"  ROC-AUC:              {auc:.4f}")
print()
print("  TOP 5 FEATURE IMPORTANCE SCORES:")
for _, row in fi.head(5).iterrows():
    print(f"    {row['Feature']:<35} {row['Importance']:.4f}")
print("=" * 65)

  FINAL MODEL PERFORMANCE — DECISION TREE
  Best hyperparameters: {'criterion': 'gini', 'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 10}
  Tree depth:           15
  Leaves:               1404

  Accuracy:             0.9386
  Precision (Satisfied): 0.9553
  Recall (Satisfied):    0.9314
  F1-Score (Satisfied):  0.9432  << primary metric
  ROC-AUC:              0.9748

  TOP 5 FEATURE IMPORTANCE SCORES:
    Inflight entertainment              0.4429
    Seat comfort                        0.1976
    Ease of Online booking              0.0659
    Departure/Arrival time convenient   0.0309
    Customer Type                       0.0288
